# Avaliação

Este notebook reúne os testes da busca e dos modelos. É a etapa mais demorada, porque cada caso passa pelos dois modelos.

Nenhum destes placares é uma validação jurídica, já que as amostras são pequenas e foram escritas à mão.

In [1]:
EXEMPLOS_JUSTIFICATIVA = False
%run ./06_justificativa_llama.ipynb

import pandas as pd

## Controle da busca

São 12 paráfrases escritas à mão, sem copiar o enunciado, cada uma com a súmula esperada. O teste serve para perceber se a busca piorou depois de uma mudança. A amostra é pequena e os limiares foram ajustados com ela, então uma avaliação de verdade precisaria de ementas reais anotadas por especialistas.

In [2]:
CONTROLE = [
    (7, "Agravo não conhecido: a análise da pretensão recursal exige revolvimento do conjunto fático-probatório dos autos, inviável na via do especial."),
    (231, "Dosimetria. Reconhecida a confissão espontânea, a pena permaneceu no piso legal, pois a segunda fase não admite fixação aquém do mínimo cominado."),
    (479, "Banco responde objetivamente pelos danos decorrentes de golpe praticado por estelionatário em conta do correntista, por se tratar de fortuito interno."),
    (443, "Roubo com duas majorantes. O acréscimo na terceira etapa da dosimetria não pode decorrer apenas da quantidade de causas de aumento, exigindo motivação concreta."),
    (297, "Contrato bancário. Relação entre correntista e instituição financeira sujeita-se às normas consumeristas."),
    (385, "Negativação indevida. Autor que já possuía outras inscrições legítimas nos órgãos de restrição ao crédito não faz jus à indenização por abalo moral, cabendo apenas o cancelamento."),
    (668, "Crime de posse de arma com numeração suprimida, de calibre permitido, não possui natureza hedionda."),
    (593, "Conjunção carnal com menor de quatorze anos. Consentimento da vítima e relacionamento amoroso anterior são irrelevantes para a tipificação."),
    (54, "Responsabilidade extracontratual: acidente de trânsito. Os juros de mora fluem a partir da data do fato."),
    (676, "Audiência de custódia. Sem pedido do Parquet ou representação do delegado, o magistrado transformou o flagrante em custódia cautelar por iniciativa própria; ilegalidade reconhecida."),
    (630, "Tráfico. Réu admitiu possuir a droga para consumo pessoal; atenuante deve ser aplicada em fração menor do que na confissão integral."),
    (601, "Ministério Público tem legitimidade ativa para ação civil pública em defesa de direitos do consumidor decorrentes da prestação de serviço público."),
]  # fmt: skip


def avaliar(limite: int = 5) -> pd.DataFrame:
    linhas = []
    for numero, texto in CONTROLE:
        citacoes = [c.citacao for c in buscar(texto, _filtro(fonte="sumula", vigente=True), limite)]
        alvo = f"Súmula {numero}/STJ"
        linhas.append({"esperada": alvo, "posição": citacoes.index(alvo) + 1 if alvo in citacoes else None, "1º resultado": citacoes[0] if citacoes else None})
    return pd.DataFrame(linhas)


avaliacao = avaliar()
display(avaliacao)
for k in (1, 3, 5):
    print(f"Acerto no top {k}: {(avaliacao['posição'] <= k).sum()}/{len(avaliacao)}")

,esperada,posição,1º resultado
0,Súmula 7/STJ,NaN,Súmula 182/STJ
1,Súmula 231/STJ,2.0,Súmula 630/STJ
2,Súmula 479/STJ,1.0,Súmula 479/STJ
3,Súmula 443/STJ,1.0,Súmula 443/STJ
4,Súmula 297/STJ,2.0,Súmula 566/STJ
5,Súmula 385/STJ,1.0,Súmula 385/STJ
6,Súmula 668/STJ,1.0,Súmula 668/STJ
7,Súmula 593/STJ,1.0,Súmula 593/STJ
8,Súmula 54/STJ,1.0,Súmula 54/STJ
9,Súmula 676/STJ,1.0,Súmula 676/STJ


Acerto no top 1: 9/12
Acerto no top 3: 11/12
Acerto no top 5: 11/12


## Gabarito dos modelos

São onze pares com a relação esperada, escritos à mão. Alguns usam ementas inventadas com a conclusão oposta à da súmula, que devem dar `DIVERGENTE`.

O prompt foi escolhido olhando estes mesmos casos, então o placar é otimista. Ele serve para comparar os modelos, não para medir a qualidade em uso real.

In [3]:
EMENTA_PEDIDO = limpar_query(QUERY_EXEMPLO)
EMENTA_CONTRARIA = "PRISÃO PREVENTIVA. CONVERSÃO DE OFÍCIO DA PRISÃO EM FLAGRANTE. POSSIBILIDADE. O juiz pode, sem requerimento do Ministério Público ou representação da autoridade policial, converter de ofício a prisão em flagrante em prisão preventiva, mesmo após a Lei n. 13.964/2019."
EMENTA_BANCO = "Banco responde objetivamente pelos danos decorrentes de golpe praticado por estelionatário em conta do correntista, por se tratar de fortuito interno."
EMENTA_NEGATIVACAO = "Negativação indevida. Ainda que o autor possua outras inscrições legítimas e anteriores nos órgãos de restrição ao crédito, é devida indenização por dano moral."
EMENTA_JUROS = "Responsabilidade extracontratual: acidente de trânsito. Os juros de mora fluem a partir da data do fato."

GABARITO = [
    (EMENTA_PEDIDO, "Súmula 676/STJ", "CONSONANTE"),
    (EMENTA_PEDIDO, "Súmula 409/STJ", "SEM CONSONÂNCIA"),
    (EMENTA_PEDIDO, "Súmula 668/STJ", "SEM CONSONÂNCIA"),
    (EMENTA_PEDIDO, "Tema 1098/STJ", "SEM CONSONÂNCIA"),
    (EMENTA_PEDIDO, "Tema 134/STJ", "SEM CONSONÂNCIA"),
    (EMENTA_CONTRARIA, "Súmula 676/STJ", "DIVERGENTE"),
    (EMENTA_BANCO, "Súmula 479/STJ", "CONSONANTE"),
    (EMENTA_BANCO, "Súmula 54/STJ", "SEM CONSONÂNCIA"),
    (EMENTA_NEGATIVACAO, "Súmula 385/STJ", "DIVERGENTE"),
    (EMENTA_JUROS, "Súmula 54/STJ", "CONSONANTE"),
    (EMENTA_JUROS, "Súmula 479/STJ", "SEM CONSONÂNCIA"),
]  # fmt: skip


def enunciado_de(citacao: str) -> str:
    pontos, _ = qdrant.scroll(
        COLECAO, scroll_filter=_filtro(citacao=citacao), limit=1, with_payload=True, with_vectors=False
    )
    return pontos[0].payload["enunciado"]


linhas_gabarito = []
respostas = []
for ementa, citacao, esperado in GABARITO:
    linha = {"ementa": ementa[:45] + "...", "precedente": citacao, "esperado": esperado}
    for modelo in (MODELO_LLM, MODELO_COMPARACAO):
        parecer = justificar(ementa, citacao, enunciado_de(citacao), modelo)
        linha[modelo] = parecer["relacao"]
        respostas.append({"modelo": modelo, **parecer})
    linhas_gabarito.append(linha)

gabarito = pd.DataFrame(linhas_gabarito)
display(gabarito)
for modelo in (MODELO_LLM, MODELO_COMPARACAO):
    acertos = (gabarito[modelo] == gabarito["esperado"]).sum()
    print(f"{modelo}: {acertos}/{len(gabarito)} | respostas: {gabarito[modelo].value_counts().to_dict()}")

,ementa,precedente,esperado,llama2:7b,llama3.1:8b
0,PRISÃO PREVENTIVA. PORTE ILEGAL DE ARMA DE FO...,Súmula 676/STJ,CONSONANTE,CONSONANTE,CONSONANTE
1,PRISÃO PREVENTIVA. PORTE ILEGAL DE ARMA DE FO...,Súmula 409/STJ,SEM CONSONÂNCIA,CONSONANTE,SEM CONSONÂNCIA
2,PRISÃO PREVENTIVA. PORTE ILEGAL DE ARMA DE FO...,Súmula 668/STJ,SEM CONSONÂNCIA,CONSONANTE,SEM CONSONÂNCIA
3,PRISÃO PREVENTIVA. PORTE ILEGAL DE ARMA DE FO...,Tema 1098/STJ,SEM CONSONÂNCIA,CONSONANTE,SEM CONSONÂNCIA
4,PRISÃO PREVENTIVA. PORTE ILEGAL DE ARMA DE FO...,Tema 134/STJ,SEM CONSONÂNCIA,CONSONANTE,SEM CONSONÂNCIA
5,PRISÃO PREVENTIVA. CONVERSÃO DE OFÍCIO DA PRI...,Súmula 676/STJ,DIVERGENTE,CONSONANTE,DIVERGENTE
6,Banco responde objetivamente pelos danos deco...,Súmula 479/STJ,CONSONANTE,CONSONANTE,CONSONANTE
7,Banco responde objetivamente pelos danos deco...,Súmula 54/STJ,SEM CONSONÂNCIA,CONSONANTE,SEM CONSONÂNCIA
8,Negativação indevida. Ainda que o autor possu...,Súmula 385/STJ,DIVERGENTE,CONSONANTE,DIVERGENTE
9,Responsabilidade extracontratual: acidente de...,Súmula 54/STJ,CONSONANTE,CONSONANTE,DIVERGENTE


llama2:7b: 3/11 | respostas: {'CONSONANTE': 11}
llama3.1:8b: 10/11 | respostas: {'SEM CONSONÂNCIA': 6, 'DIVERGENTE': 3, 'CONSONANTE': 2}


In [4]:
respostas_df = pd.DataFrame(respostas)
for modelo, grupo in respostas_df.groupby("modelo"):
    preenchidas = (grupo["justificativa"] != "").sum()
    print(f"{modelo}: {preenchidas}/{len(grupo)} com justificativa | {grupo['tipo_justificativa'].value_counts().to_dict()}")
display(respostas_df[["modelo", "citacao", "relacao", "tipo_justificativa", "justificativa", "alertas"]])

llama2:7b: 11/11 com justificativa | {'JURIDICA': 11}
llama3.1:8b: 11/11 com justificativa | {'JURIDICA': 8, 'FATICA': 3}


,modelo,citacao,relacao,tipo_justificativa,justificativa,alertas
0,llama2:7b,Súmula 676/STJ,CONSONANTE,JURIDICA,A decisão do precedente é baseada na interpret...,[]
1,llama3.1:8b,Súmula 676/STJ,CONSONANTE,JURIDICA,A ementa trata da impossibilidade de decretaçã...,[]
2,llama2:7b,Súmula 409/STJ,CONSONANTE,JURIDICA,A decisão do precedente é baseada na interpret...,[]
3,llama3.1:8b,Súmula 409/STJ,SEM CONSONÂNCIA,FATICA,A ementa trata de prisão preventiva e conversã...,[]
4,llama2:7b,Súmula 668/STJ,CONSONANTE,JURIDICA,A ementa decide que a conversão da prisão em f...,[]
5,llama3.1:8b,Súmula 668/STJ,SEM CONSONÂNCIA,JURIDICA,"A ementa não trata da regra do precedente, poi...",[]
6,llama2:7b,Tema 1098/STJ,CONSONANTE,JURIDICA,O princípio da retroatividade da norma pena be...,[o trecho citado não existe na ementa]
7,llama3.1:8b,Tema 1098/STJ,SEM CONSONÂNCIA,FATICA,"A ementa não trata da regra do precedente, poi...",[]
8,llama2:7b,Tema 134/STJ,CONSONANTE,JURIDICA,A decisão do precedente é baseada na interpret...,[]
9,llama3.1:8b,Tema 134/STJ,SEM CONSONÂNCIA,FATICA,A ementa trata de prisão preventiva e conversã...,[]


## Gabarito do tipo de justificativa

São oito pares com o tipo esperado. Nos três últimos a regra é a mesma, e é um fato do caso que afasta a aplicação. A definição dos tipos também foi ajustada olhando estes casos, então este placar é igualmente otimista.

In [5]:
EMENTA_IDADE = "Estupro de vulnerável afastado. A vítima tinha quinze anos completos à época dos fatos e não apresentava enfermidade ou deficiência, conforme laudo e certidão de nascimento juntados aos autos."
EMENTA_VIA_PUBLICA = "Indenização negada. O veículo do autor foi furtado quando estava estacionado na via pública, em frente à loja, pois a ré não possui estacionamento próprio."
EMENTA_RUA = "Banco não responde. A autora entregou o dinheiro em espécie, em mãos, a um desconhecido na rua, sem que houvesse qualquer operação em sua conta ou uso de serviço bancário."

GABARITO_TIPO = [
    (EMENTA_PEDIDO, "Súmula 676/STJ", "JURIDICA"),
    (EMENTA_PEDIDO, "Súmula 409/STJ", "JURIDICA"),
    (EMENTA_CONTRARIA, "Súmula 676/STJ", "JURIDICA"),
    (EMENTA_BANCO, "Súmula 479/STJ", "JURIDICA"),
    (EMENTA_BANCO, "Súmula 54/STJ", "JURIDICA"),
    (EMENTA_IDADE, "Súmula 593/STJ", "FATICA"),
    (EMENTA_VIA_PUBLICA, "Súmula 130/STJ", "FATICA"),
    (EMENTA_RUA, "Súmula 479/STJ", "FATICA"),
]  # fmt: skip

linhas_tipo = []
for ementa, citacao, esperado in GABARITO_TIPO:
    linha = {"ementa": ementa[:45] + "...", "precedente": citacao, "esperado": esperado}
    for modelo in (MODELO_LLM, MODELO_COMPARACAO):
        linha[modelo] = justificar(ementa, citacao, enunciado_de(citacao), modelo)["tipo_justificativa"]
    linhas_tipo.append(linha)

gabarito_tipo = pd.DataFrame(linhas_tipo)
display(gabarito_tipo)
for modelo in (MODELO_LLM, MODELO_COMPARACAO):
    print(f"{modelo}: {(gabarito_tipo[modelo] == gabarito_tipo['esperado']).sum()}/{len(gabarito_tipo)}")

,ementa,precedente,esperado,llama2:7b,llama3.1:8b
0,PRISÃO PREVENTIVA. PORTE ILEGAL DE ARMA DE FO...,Súmula 676/STJ,JURIDICA,JURIDICA,JURIDICA
1,PRISÃO PREVENTIVA. PORTE ILEGAL DE ARMA DE FO...,Súmula 409/STJ,JURIDICA,JURIDICA,FATICA
2,PRISÃO PREVENTIVA. CONVERSÃO DE OFÍCIO DA PRI...,Súmula 676/STJ,JURIDICA,JURIDICA,JURIDICA
3,Banco responde objetivamente pelos danos deco...,Súmula 479/STJ,JURIDICA,JURIDICA,JURIDICA
4,Banco responde objetivamente pelos danos deco...,Súmula 54/STJ,JURIDICA,JURIDICA,JURIDICA
5,Estupro de vulnerável afastado. A vítima tinh...,Súmula 593/STJ,FATICA,JURIDICA,FATICA
6,Indenização negada. O veículo do autor foi fu...,Súmula 130/STJ,FATICA,JURIDICA,JURIDICA
7,Banco não responde. A autora entregou o dinhe...,Súmula 479/STJ,FATICA,JURIDICA,JURIDICA


llama2:7b: 5/8
llama3.1:8b: 5/8
